In [ ]:
%load_ext autoreload
%autoreload 2

# 06b — Barycenter Baseline Comparison

This notebook evaluates 7 barycenter methods on the symbolic cooking sequences:

| # | Method | Type | Source |
|---|--------|------|--------|
| 1 | **TW-TWE + medoid** | Proposed distance (medoid classifier) | Precomputed `D_twe.npy` from NB06 |
| 2 | DBA + standard DTW | Ablation (no Wasserstein cost) | `baselines.barycenter_dba_dtw` |
| 3 | Soft-DTW barycenter | Differentiable alignment | `baselines.barycenter_soft_dtw` |
| 4 | Edit-distance median | Pure string (no temporal) | `baselines.barycenter_edit_median` |
| 5 | Wasserstein barycenter | Distributional (histogram) | `baselines.barycenter_wasserstein` |
| 6 | k-Medoid | No averaging (select best) | `baselines.barycenter_k_medoid` |
| 7 | Majority voting | Lock-step (no alignment) | `baselines.barycenter_majority_voting` |

**Protocol:** 10 random 50/50 stratified splits × 3 initializations. AUC-ROC for pairwise group classification.

**Prerequisites:** Run NB06 first to generate `$DATA_ROOT/outputs/symbolic_barycenter/` outputs.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score

from smartflat.utils.utils_io import get_data_root
from smartflat.features.symbolic_barycenter.baselines import (
    evaluate_baselines,
    barycenter_dba_dtw,
    barycenter_soft_dtw,
    barycenter_edit_median,
    barycenter_wasserstein,
    barycenter_k_medoid,
    barycenter_majority_voting,
    embed_symbolic_to_real,
    project_real_to_symbolic,
)

In [ ]:
# Load precomputed outputs from NB06
data_root = get_data_root()
out_dir = os.path.join(data_root, 'outputs', 'symbolic_barycenter')

D_twe = np.load(os.path.join(out_dir, 'D_twe.npy'))
D_G = np.load(os.path.join(out_dir, 'D_G.npy'))
X_aeon = np.load(os.path.join(out_dir, 'X_aeon.npy'))

with open(os.path.join(out_dir, 'barycenters.pkl'), 'rb') as f:
    barycenters_twtwe = pickle.load(f)

with open(os.path.join(out_dir, 'split_data.pkl'), 'rb') as f:
    split_data = pickle.load(f)

# Extract symbolic sequences: X_aeon is (N, 1, T) -> X_symbolic is (N, T)
X_symbolic = X_aeon[:, 0, :].astype(np.int64)
N, T = X_symbolic.shape
G = D_G.shape[0]

# Reconstruct full labels array from split_data
labels = np.empty(N, dtype=object)
labels[split_data['train_idx']] = split_data['pathologie_train']
labels[split_data['test_idx']] = split_data['pathologie_test']
assert not any(l is None for l in labels), "Missing labels"

print(f"Sequences: {N} × {T} symbols (G={G})")
print(f"D_twe: {D_twe.shape}")
print(f"Groups: {dict(zip(*np.unique(labels, return_counts=True)))}")
print(f"Precomputed barycenters: {list(barycenters_twtwe.keys())}")

### Baseline evaluation

**Protocol:** 10 random 50/50 stratified train/test splits × 3 random initializations per split.
For each split, barycenters are computed on training data only. Test sequences are classified
by distance to nearest group barycenter. AUC-ROC is computed for each pairwise group comparison.

**Runtime note:** DBA-DTW and Soft-DTW operate on sequences of length ~5000 and may take
30–60 minutes for the full evaluation. Set `QUICK_MODE = True` for fast iteration (2 splits, 1 init).

In [ ]:
QUICK_MODE = False  # Set True for fast iteration (2 splits, 1 init)

n_splits = 2 if QUICK_MODE else 10
n_inits = 1 if QUICK_MODE else 3

methods = {
    'dba_dtw': barycenter_dba_dtw,
    'soft_dtw': barycenter_soft_dtw,
    'edit_median': lambda X, D_G, random_state=None: barycenter_edit_median(
        X, n_alphabet=D_G.shape[0],
    ),
    'wasserstein': barycenter_wasserstein,
    'k_medoid': barycenter_k_medoid,
    'majority_voting': barycenter_majority_voting,
}

print(f"Running {len(methods)} baselines: {list(methods.keys())}")
print(f"Protocol: {n_splits} splits × {n_inits} inits")

df_baselines = evaluate_baselines(
    X_symbolic, labels, D_G, D_twe,
    methods=methods,
    n_splits=n_splits,
    n_inits=n_inits,
    random_state=42,
)

print(f"\nBaseline results: {len(df_baselines)} rows")
display(df_baselines.groupby(['method', 'comparison'])['auc'].agg(['mean', 'std']).round(3))

In [ ]:
# TW-TWE evaluation: medoid classification using precomputed D_twe
# Uses the same split protocol as evaluate_baselines() for fair comparison.
# No aeon dependency — only needs the distance matrix.

unique_groups = sorted(np.unique(labels))
splitter = StratifiedShuffleSplit(
    n_splits=n_splits, test_size=0.5, random_state=42,
)

twtwe_records = []
for split_idx, (train_idx, test_idx) in enumerate(splitter.split(X_symbolic, labels)):
    assert len(set(train_idx) & set(test_idx)) == 0

    # Find per-group medoid in TW-TWE space
    medoid_indices = {}
    for grp in unique_groups:
        grp_train = train_idx[labels[train_idx] == grp]
        D_sub = D_twe[np.ix_(grp_train, grp_train)]
        medoid_indices[grp] = grp_train[np.argmin(D_sub.sum(axis=1))]

    # Classify test sequences by distance to nearest group medoid
    test_dists = np.zeros((len(test_idx), len(unique_groups)))
    for gi, grp in enumerate(unique_groups):
        test_dists[:, gi] = D_twe[test_idx, medoid_indices[grp]]

    # Pairwise AUC-ROC
    test_labels = labels[test_idx]
    for g1_idx, g1 in enumerate(unique_groups):
        for g2_idx, g2 in enumerate(unique_groups):
            if g1_idx >= g2_idx:
                continue
            mask = np.isin(test_labels, [g1, g2])
            if mask.sum() < 4:
                continue
            y_true = (test_labels[mask] == g2).astype(int)
            y_score = test_dists[mask, g1_idx] - test_dists[mask, g2_idx]
            try:
                auc = roc_auc_score(y_true, y_score)
            except ValueError:
                auc = 0.5
            twtwe_records.append({
                'method': 'tw_twe',
                'split': split_idx,
                'init': 0,
                'comparison': f'{g1}_vs_{g2}',
                'auc': auc,
            })

df_twtwe = pd.DataFrame(twtwe_records)
print(f"TW-TWE results: {len(df_twtwe)} rows")
display(df_twtwe.groupby('comparison')['auc'].agg(['mean', 'std']).round(3))

In [ ]:
# Combine all results
df_all = pd.concat([df_baselines, df_twtwe], ignore_index=True)

# Summary table
summary = (
    df_all.groupby(['method', 'comparison'])['auc']
    .agg(['mean', 'std', 'count'])
    .round(3)
)
print(f"Full comparison: {df_all['method'].nunique()} methods × {df_all['comparison'].nunique()} comparisons")
display(summary)

### AUC comparison figure

In [ ]:
# Grouped bar chart: AUC by method and comparison pair
auc_summary = (
    df_all.groupby(['method', 'comparison'])['auc']
    .agg(['mean', 'std'])
    .reset_index()
)

# Order methods: proposed first, then baselines alphabetically
method_order = ['tw_twe'] + sorted([m for m in df_all['method'].unique() if m != 'tw_twe'])
comparison_order = sorted(df_all['comparison'].unique())

fig, ax = plt.subplots(figsize=(12, 5))
n_methods = len(method_order)
n_comparisons = len(comparison_order)
bar_width = 0.8 / n_methods

colors = plt.cm.Set2(np.linspace(0, 1, n_methods))

for mi, method in enumerate(method_order):
    means, stds, xs = [], [], []
    for ci, comp in enumerate(comparison_order):
        row = auc_summary[(auc_summary['method'] == method) & (auc_summary['comparison'] == comp)]
        if len(row) > 0:
            means.append(row['mean'].values[0])
            stds.append(row['std'].values[0])
        else:
            means.append(0)
            stds.append(0)
        xs.append(ci + mi * bar_width)
    ax.bar(xs, means, bar_width, yerr=stds, label=method, color=colors[mi],
           edgecolor='white', linewidth=0.5, capsize=2)

ax.set_xticks([ci + (n_methods - 1) * bar_width / 2 for ci in range(n_comparisons)])
ax.set_xticklabels([c.replace('_', ' ') for c in comparison_order])
ax.set_ylabel('AUC-ROC')
ax.set_title('Barycenter Method Comparison — Classification AUC')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='chance')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.set_ylim(0, 1.05)
fig.tight_layout()

fig.savefig(os.path.join(out_dir, 'baseline_comparison_auc.png'), dpi=150, bbox_inches='tight')
print(f"Saved: {os.path.join(out_dir, 'baseline_comparison_auc.png')}")
plt.show()

### Barycenter chronogram visualization

Example barycenters for each method (computed on the first split's training set).
Wasserstein produces a histogram rather than a sequence, shown separately.

In [ ]:
# Compute one example barycenter per group per method (split 0, seed 42)
splitter_ex = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
train_idx_ex, _ = next(splitter_ex.split(X_symbolic, labels))
seed = 42

groups = sorted(np.unique(labels))
cmap = plt.cm.get_cmap('tab20', G)

# Sequence-based methods
seq_methods = {
    'dba_dtw': lambda X, grp: barycenter_dba_dtw(X, D_G, random_state=seed),
    'soft_dtw': lambda X, grp: barycenter_soft_dtw(X, D_G, random_state=seed),
    'edit_median': lambda X, grp: barycenter_edit_median(X, n_alphabet=G),
    'majority_voting': lambda X, grp: barycenter_majority_voting(X),
}

fig, axes = plt.subplots(
    len(seq_methods) + 1, len(groups),
    figsize=(20, 2.5 * (len(seq_methods) + 1)),
    sharex=True,
)

# Row 0: TW-TWE (precomputed from NB06)
for gi, grp in enumerate(groups):
    b = barycenters_twtwe[grp].squeeze()
    axes[0, gi].imshow(
        b[np.newaxis, :], aspect='auto', cmap=cmap, vmin=0, vmax=G - 1,
        interpolation='nearest',
    )
    axes[0, gi].set_yticks([])
    if gi == 0:
        axes[0, gi].set_ylabel('TW-TWE\n(DBA)', fontsize=9)
    axes[0, gi].set_title(grp, fontweight='bold')

# Rows 1+: baseline methods
for mi, (method_name, method_fn) in enumerate(seq_methods.items(), start=1):
    for gi, grp in enumerate(groups):
        grp_mask = labels[train_idx_ex] == grp
        X_grp = X_symbolic[train_idx_ex][grp_mask]
        b = method_fn(X_grp, grp)
        axes[mi, gi].imshow(
            b[np.newaxis, :min(len(b), T)], aspect='auto', cmap=cmap,
            vmin=0, vmax=G - 1, interpolation='nearest',
        )
        axes[mi, gi].set_yticks([])
        if gi == 0:
            axes[mi, gi].set_ylabel(method_name.replace('_', '\n'), fontsize=9)

fig.suptitle('Barycenter Chronograms by Method and Group', fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, 'baseline_chronograms.png'), dpi=150, bbox_inches='tight')
print(f"Saved: {os.path.join(out_dir, 'baseline_chronograms.png')}")
plt.show()

# Wasserstein: histogram visualization
fig_w, axes_w = plt.subplots(1, len(groups), figsize=(15, 3), sharey=True)
for gi, grp in enumerate(groups):
    grp_mask = labels[train_idx_ex] == grp
    X_grp = X_symbolic[train_idx_ex][grp_mask]
    h = barycenter_wasserstein(X_grp, D_G)
    axes_w[gi].bar(range(G), h, color=[cmap(i) for i in range(G)], edgecolor='white', linewidth=0.3)
    axes_w[gi].set_title(grp, fontweight='bold')
    axes_w[gi].set_xlabel('Symbol')
    if gi == 0:
        axes_w[gi].set_ylabel('Frequency')
fig_w.suptitle('Wasserstein Barycenter (Histogram)', fontweight='bold')
fig_w.tight_layout()
fig_w.savefig(os.path.join(out_dir, 'baseline_wasserstein_histograms.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save comparison results
csv_path = os.path.join(out_dir, 'baseline_comparison.csv')
df_all.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
print(f"  {len(df_all)} rows, {df_all['method'].nunique()} methods")

# Summary for paper
print("\n=== Summary for paper ===")
pivot = (
    df_all.groupby(['method', 'comparison'])['auc']
    .agg(['mean', 'std'])
    .round(3)
    .unstack('comparison')
)
display(pivot)